In [ ]:
pip install gensim scikit-learn


## Importação Bibliotecas


In [ ]:
import pandas as pd
import numpy as np
import gensim
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from gensim.models import Word2Vec
from sklearn.pipeline import Pipeline

# Arcaico vs Moderno

In [ ]:
df = pd.read_csv("train_arcaico_moderno.csv", sep=";")
Y=df['style']

In [ ]:
X= df['text']
Y=df['style']

x_train, x_test, y_train, y_test = train_test_split(X,Y, test_size=0.2,
random_state=123, stratify=Y)


## Bag of Words

In [ ]:
pipeline = Pipeline([
    ('vect', CountVectorizer()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000)),
])

parameters = {
    # Testa ignorar palavras que aparecem em mais de 50% ou 75% dos docs
    'vect__max_df': [0.5, 0.75, 1.0],

    # Testa ignorar palavras que aparecem em menos de 2, 5 ou 10 docs
    'vect__min_df': [2, 5, 10],

    # Testa diferentes valores de regularização para a Regressão Logística
    'clf__C': [0.1, 1, 10],
}

grid_search = GridSearchCV(pipeline, parameters, cv=10, scoring='f1_macro', n_jobs=-1, verbose=1)

print("Iniciando o Grid Search para o BoW puro... Isso pode demorar um pouco.")
grid_search.fit(x_train, y_train)


print("\n\n--- Resultados do Grid Search ---")
print("Melhores parâmetros encontrados para o BoW puro:")
print(grid_search.best_params_)

print("\nMelhor F1-Score (macro) durante a validação cruzada:")
print("{:2.2f}".format(grid_search.best_score_).replace(".", ","))

Iniciando o Grid Search para o BoW puro... Isso pode demorar um pouco.
Fitting 5 folds for each of 27 candidates, totalling 135 fits


--- Resultados do Grid Search ---
Melhores parâmetros encontrados para o BoW puro:
{'clf__C': 1, 'vect__max_df': 0.5, 'vect__min_df': 2}

Melhor F1-Score (macro) durante a validação cruzada:
0,83


In [ ]:
print("\n\n--- Avaliação Final no Conjunto de Teste ---")
best_model = grid_search.best_estimator_
predicted = best_model.predict(x_test)

score = f1_score(y_test, predicted, average='macro')
print("\n\nBoW Puro Otimizado (Grid Search) =====>{:2.2f}\n\n".format(score).replace(".", ","))

print("Relatório de Classificação Detalhado:")
print(classification_report(y_test, predicted))



--- Avaliação Final no Conjunto de Teste ---


BoW Puro Otimizado (Grid Search) =====>0,83


Relatório de Classificação Detalhado:
              precision    recall  f1-score   support

     arcaico       0.83      0.82      0.83      3689
     moderno       0.83      0.84      0.83      3688

    accuracy                           0.83      7377
   macro avg       0.83      0.83      0.83      7377
weighted avg       0.83      0.83      0.83      7377



## Regressão Logística sobre contagens TF-IDF

In [ ]:
pipeline = Pipeline([
    ('vect', TfidfVectorizer()), # <-- MUDANÇA AQUI
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000)),
])

parameters = {
    'vect__max_df': [0.5, 0.75, 1.0],
    'vect__min_df': [2, 5, 10],


    'vect__use_idf': [True, False],
    'vect__norm': ['l1', 'l2'],


    'clf__C': [0.1, 1, 10],
}



grid_search = GridSearchCV(pipeline, parameters, cv=10, scoring='f1_macro', n_jobs=-1, verbose=1)

print("Iniciando o Grid Search para o TF-IDF... Isso pode demorar um pouco.")
grid_search.fit(x_train, y_train)


print("\n\n--- Resultados do Grid Search ---")
print("Melhores parâmetros encontrados para o TF-IDF:")
print(grid_search.best_params_)

print("\nMelhor F1-Score (macro) durante a validação cruzada:")
print("{:2.2f}".format(grid_search.best_score_).replace(".", ","))

Iniciando o Grid Search para o TF-IDF... Isso pode demorar um pouco.
Fitting 5 folds for each of 108 candidates, totalling 540 fits


--- Resultados do Grid Search ---
Melhores parâmetros encontrados para o TF-IDF:
{'clf__C': 10, 'vect__max_df': 0.75, 'vect__min_df': 2, 'vect__norm': 'l2', 'vect__use_idf': True}

Melhor F1-Score (macro) durante a validação cruzada:
0,83


In [ ]:
print("\n\n--- Avaliação Final no Conjunto de Teste ---")
best_model = grid_search.best_estimator_
predicted = best_model.predict(x_test)

score = f1_score(y_test, predicted, average='macro')
print("\n\nTF-IDF Otimizado (Grid Search) =====>{:2.2f}\n\n".format(score).replace(".", ","))

print("Relatório de Classificação Detalhado:")
print(classification_report(y_test, predicted))



--- Avaliação Final no Conjunto de Teste ---


TF-IDF Otimizado (Grid Search) =====>0,83


Relatório de Classificação Detalhado:
              precision    recall  f1-score   support

     arcaico       0.83      0.82      0.83      3689
     moderno       0.83      0.84      0.83      3688

    accuracy                           0.83      7377
   macro avg       0.83      0.83      0.83      7377
weighted avg       0.83      0.83      0.83      7377



## N-gramas

In [ ]:

parameters = {

    'vect__ngram_range': [(1, 1), (1, 2)],

    'vect__max_df': [0.5, 0.75],
    'vect__min_df': [2, 5],

    'vect__use_idf': [True, False],
    'vect__norm': ['l1', 'l2'],

    'clf__C': [0.1, 1, 10],
}



grid_search = GridSearchCV(pipeline, parameters, cv=10, scoring='f1_macro', n_jobs=-1, verbose=1)

print("Iniciando o Grid Search para TF-IDF com N-gramas... Isso pode ser o mais demorado.")
grid_search.fit(x_train, y_train)


# 5. ANALISAR OS RESULTADOS
print("\n\n--- Resultados do Grid Search ---")
print("Melhores parâmetros encontrados para TF-IDF + N-gramas:")
print(grid_search.best_params_)

print("\nMelhor F1-Score (macro) durante a validação cruzada:")
print("{:2.2f}".format(grid_search.best_score_).replace(".", ","))

Iniciando o Grid Search para TF-IDF com N-gramas... Isso pode ser o mais demorado.
Fitting 10 folds for each of 96 candidates, totalling 960 fits


--- Resultados do Grid Search ---
Melhores parâmetros encontrados para TF-IDF + N-gramas:
{'clf__C': 10, 'vect__max_df': 0.5, 'vect__min_df': 2, 'vect__ngram_range': (1, 2), 'vect__norm': 'l2', 'vect__use_idf': True}

Melhor F1-Score (macro) durante a validação cruzada:
0,85


In [ ]:
print("\n\n--- Avaliação Final no Conjunto de Teste ---")
best_model = grid_search.best_estimator_
predicted = best_model.predict(x_test)

score = f1_score(y_test, predicted, average='macro')
print("\n\nTF-IDF + N-gramas (Otimizado) =====>{:2.2f}\n\n".format(score).replace(".", ","))

print("Relatório de Classificação Detalhado:")
print(classification_report(y_test, predicted))



--- Avaliação Final no Conjunto de Teste ---


TF-IDF + N-gramas (Otimizado) =====>0,85


Relatório de Classificação Detalhado:
              precision    recall  f1-score   support

     arcaico       0.85      0.84      0.85      3689
     moderno       0.84      0.85      0.85      3688

    accuracy                           0.85      7377
   macro avg       0.85      0.85      0.85      7377
weighted avg       0.85      0.85      0.85      7377



## Seleção de Atributos

In [ ]:
pipeline = Pipeline([
    ('vect', TfidfVectorizer()),
    ('select', SelectKBest(score_func=chi2)),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000)),
])


k_values = list(range(30000, 3000, -3000)) + ['all']
print(f"Valores de 'k' que serão testados para seleção de atributos: {k_values}")

parameters = {
    'vect__ngram_range': [(1, 1), (1, 2)],
    'vect__min_df': [3, 5],

    'select__k': k_values,

    'clf__C': [1, 10],
}



grid_search = GridSearchCV(pipeline, parameters, cv=10, scoring='f1_macro', n_jobs=-1, verbose=2)

print("\nIniciando o Grid Search com seleção de atributos (30k a 6k)...")
grid_search.fit(x_train, y_train)



print("\n\n--- Resultados do Grid Search ---")
print("Melhores parâmetros encontrados:")
print(grid_search.best_params_)

print("\nMelhor F1-Score (macro) durante a validação cruzada:")
print("{:2.2f}".format(grid_search.best_score_).replace(".", ","))





Valores de 'k' que serão testados para seleção de atributos: [30000, 27000, 24000, 21000, 18000, 15000, 12000, 9000, 6000, 'all']

Iniciando o Grid Search com seleção de atributos (30k a 6k)...
Fitting 10 folds for each of 80 candidates, totalling 800 fits


--- Resultados do Grid Search ---
Melhores parâmetros encontrados:
{'clf__C': 10, 'select__k': 18000, 'vect__min_df': 3, 'vect__ngram_range': (1, 2)}

Melhor F1-Score (macro) durante a validação cruzada:
0,85


In [ ]:
print("\n\n--- Avaliação Final no Conjunto de Teste ---")
best_model = grid_search.best_estimator_
predicted = best_model.predict(x_test)

score = f1_score(y_test, predicted, average='macro')
print("\n\nModelo Final Otimizado (com Seleção de Atributos) =====>{:2.2f}\n\n".format(score).replace(".", ","))

print("Relatório de Classificação Detalhado:")
print(classification_report(y_test, predicted))



--- Avaliação Final no Conjunto de Teste ---


Modelo Final Otimizado (com Seleção de Atributos) =====>0,86


Relatório de Classificação Detalhado:
              precision    recall  f1-score   support

     arcaico       0.87      0.85      0.86      3689
     moderno       0.85      0.87      0.86      3688

    accuracy                           0.86      7377
   macro avg       0.86      0.86      0.86      7377
weighted avg       0.86      0.86      0.86      7377



## Word2Vec

In [ ]:

# Tokenizar os textos para o Word2Vec
tokenized_text = [text.lower().split() for text in df['text']]

# Treinar o modelo Word2Vec
print("Treinando o modelo Word2Vec...")
w2v_model = gensim.models.Word2Vec(sentences=tokenized_text,
                                   vector_size=100,
                                   window=5,
                                   min_count=2,
                                   workers=4)
w2v_model.init_sims(replace=True)

def document_vectorizer(text, model):
    doc_tokens = text.lower().split()
    vectors = [model.wv[token] for token in doc_tokens if token in model.wv]
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(model.vector_size)

print("Criando os vetores dos documentos...")
X = np.array([document_vectorizer(text, w2v_model) for text in df['text']])
Y = df['style']



x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2,
                                                    random_state=123, stratify=Y)

clf = LogisticRegression(class_weight='balanced', max_iter=2000)


parameters = {
    'C': [0.01, 0.1, 1, 10, 100],            # Testa diferentes forças de regularização
    'solver': ['liblinear', 'saga'],         # Testa diferentes algoritmos de otimização
    'penalty': ['l1', 'l2']                  # Testa diferentes tipos de regularização
}


grid_search = GridSearchCV(estimator=clf,
                           param_grid=parameters,
                           cv=10,
                           scoring='f1_macro',
                           n_jobs=-1,
                           verbose=1)

print("\nIniciando Grid Search para o classificador com vetores Word2Vec...")
grid_search.fit(x_train, y_train)



print("\n\n--- Resultados do Grid Search ---")
print("Melhores parâmetros encontrados para o classificador:")
print(grid_search.best_params_)

print("\nMelhor F1-Score (macro) durante a validação cruzada:")
print("{:2.2f}".format(grid_search.best_score_).replace(".", ","))


Treinando o modelo Word2Vec...


/tmp/ipython-input-2380785949.py:11: DeprecationWarning: Call to deprecated `init_sims` (Gensim 4.0.0 implemented internal optimizations that make calls to init_sims() unnecessary. init_sims() is now obsoleted and will be completely removed in future versions. See https://github.com/RaRe-Technologies/gensim/wiki/Migrating-from-Gensim-3.x-to-4).
  w2v_model.init_sims(replace=True)


Criando os vetores dos documentos...

Iniciando Grid Search para o classificador com vetores Word2Vec...
Fitting 10 folds for each of 20 candidates, totalling 200 fits


--- Resultados do Grid Search ---
Melhores parâmetros encontrados para o classificador:
{'C': 100, 'penalty': 'l1', 'solver': 'saga'}

Melhor F1-Score (macro) durante a validação cruzada:
0,69


In [ ]:

# Avaliação final usando o melhor modelo encontrado
print("\n\n--- Avaliação Final no Conjunto de Teste ---")
best_model = grid_search.best_estimator_
predicted = best_model.predict(x_test)

score = f1_score(y_test, predicted, average='macro')
print("\n\nWord2Vec Otimizado (Grid Search) =====>{:2.2f}\n\n".format(score).replace(".", ","))

print("Relatório de Classificação Detalhado:")
print(classification_report(y_test, predicted))



--- Avaliação Final no Conjunto de Teste ---


Word2Vec Otimizado (Grid Search) =====>0,69


Relatório de Classificação Detalhado:
              precision    recall  f1-score   support

     arcaico       0.69      0.67      0.68      3689
     moderno       0.68      0.70      0.69      3688

    accuracy                           0.69      7377
   macro avg       0.69      0.69      0.69      7377
weighted avg       0.69      0.69      0.69      7377



## Teste do melhor modelo encontrado em outro csv


In [ ]:
df = pd.read_csv("train_literal_dinamico.csv", sep=";")

# Remove ou substitui valores nulos
df['text'] = df['text'].fillna("")  # substitui NaN por string vazia

X = df['text']
Y = df['style']

x_train, x_test, y_train, y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=123,
    stratify=Y
)
print(X)


0        E João dá aqui testemunho das palavras de Deus...
1        Enquanto falava, os especialistas na Lei e os ...
2        E circuncidareis a carne do vosso prepúcio; e ...
3        E sucedeu que, no dia seguinte, o povo pela ma...
4        E aconteceu que, naquele dia, quando o Senhor ...
                               ...                        
36959    Ela padecera muito sob o cuidado de vários méd...
36960    somos tratados como desconhecidos, embora seja...
36961    Disse-lhes: “Está escrito: ‘ A minha casa será...
36962    “Porque assim diz o Soberano, o Senhor: Eu mes...
36963    Qualquer pessoa que comer alguma coisa no Dia ...
Name: text, Length: 36964, dtype: object


In [ ]:

final_model_pipeline = Pipeline([
    ('vect', TfidfVectorizer(
        min_df=3,
        ngram_range=(1, 2)
    )),
    ('select', SelectKBest(
        score_func=chi2,
        k=18000
    )),
    ('clf', LogisticRegression(
        C=10,
        class_weight='balanced',
        max_iter=2000, # Aumentado para garantir a convergência
        random_state=123
    )),
])

print("Pipeline construído com sucesso.")



print("Treinando o modelo final no conjunto de treino...")
final_model_pipeline.fit(x_train, y_train)
print("Treinamento concluído.")



Pipeline construído com sucesso.
Treinando o modelo final no conjunto de treino...
Treinamento concluído.


In [ ]:

print("\n--- Avaliação Final no Conjunto de Teste ---")
predicted = final_model_pipeline.predict(x_test)


score = f1_score(y_test, predicted, average='macro')
print("\n\nModelo Final Otimizado =====>{:2.2f}\n\n".format(score).replace(".", ","))


print("Relatório de Classificação Detalhado:")
print(classification_report(y_test, predicted))


--- Avaliação Final no Conjunto de Teste ---


Modelo Final Otimizado =====>0,86


Relatório de Classificação Detalhado:
              precision    recall  f1-score   support

    dinamico       0.86      0.87      0.86      3697
     literal       0.86      0.86      0.86      3696

    accuracy                           0.86      7393
   macro avg       0.86      0.86      0.86      7393
weighted avg       0.86      0.86      0.86      7393

